In [ ]:
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
from pathsnap_candidates import FEATURES, generate_candidates_from_files

with ZipFile("/content/Student_folder.zip") as archive:
    archive.extractall("pathsnap_data")

train_dir = Path("pathsnap_data/Student folder/train")
references = pd.read_csv("train_references.csv", dtype={"trip_id": str})
references_by_trip = references.groupby("trip_id")

output = Path("train_candidates.csv")
for index, trip_dir in enumerate(sorted(p for p in train_dir.iterdir() if p.is_dir())):
    trip_id = trip_dir.name
    track, nodes, arcs = (trip_dir / f"{trip_id}.{ext}" for ext in ("track", "nodes", "arcs"))
    candidates = generate_candidates_from_files(track, nodes, arcs)

    labels = references_by_trip.get_group(trip_id).dropna(subset=["reference_arc_id"])
    candidates = candidates.merge(
        labels[["observation_index", "reference_arc_id"]],
        on="observation_index",
        how="inner",
        validate="many_to_one",
    )
    candidates.insert(0, "trip_id", trip_id)
    candidates["label"] = candidates["arc_id"].eq(candidates["reference_arc_id"]).astype("int8")
    candidates[["trip_id", "observation_index", "arc_id", *FEATURES, "label"]].to_csv(
        output, mode="w" if index == 0 else "a", header=index == 0, index=False
    )

print("Saved:", output)
print("Feature columns:", FEATURES)

Saved: train_candidates.csv
Feature columns: ['dist_m', 'dist_min_m', 'dist_rel_m', 'rank_dist', 'n_cands', 'proj_frac', 'arc_len_m', 'dist_end_m', 'head_diff_deg', 'head_cos', 'speed_mps', 'moving', 'dt_prev_s', 'dist_prev_pt_m', 'dist_next_pt_m', 'has_reverse', 'deg_from', 'deg_to']


In [ ]:
from pathlib import Path
from zipfile import ZipFile, is_zipfile

p = Path("/content/Student_folder.zip")
print("Size:", f"{p.stat().st_size:,}", "bytes")
print("Valid ZIP:", is_zipfile(p))

if is_zipfile(p):
    with ZipFile(p) as z:
        print("Corrupt entry:", z.testzip())

Size: 29,410,509 bytes
Valid ZIP: True
Corrupt entry: None


In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/train_candidates.csv",
    usecols=["trip_id", "observation_index", "arc_id"],
    dtype={"trip_id": str},
)
print("Trips in CSV:", df["trip_id"].nunique(), "/ 80")
print("Rows with missing IDs:", df.isna().any(axis=1).sum())

Trips in CSV: 80 / 80
Rows with missing IDs: 0
